<a href="https://colab.research.google.com/github/Aditi-k1912/NNDL/blob/main/PRACT9_NNDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q kagglehub

# 2. Import libraries
import kagglehub
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from sklearn.model_selection import train_test_split
import os

path = kagglehub.dataset_download("rehanliaqat17/imbd-dataset")
print("Path to dataset files:", path)
print("Files:", os.listdir(path))

# Dynamically resolve file path in the downloaded directory
csv_path = os.path.join(path, "IMDB Dataset.csv")
if not os.path.exists(csv_path):
    # Fallback to search for any csv if name is slightly different
    csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
    if csv_files:
        csv_path = os.path.join(path, csv_files[0])

df = pd.read_csv(csv_path)

print(df.head())
print("Dataset size:", df.shape)
df["sentiment"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["sentiment"],
    test_size=0.2,
    random_state=42
)

tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

X_train = pad_sequences(X_train, maxlen=200)
X_test = pad_sequences(X_test, maxlen=200)

model = Sequential([
    Embedding(10000, 32),
    SimpleRNN(32),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=64
)

loss, accuracy = model.evaluate(X_test, y_test)
print("\nTest Accuracy:", accuracy * 100, "%")

def predict_sentiment(sentence):
    # Convert sentence into numbers
    sequence = tokenizer.texts_to_sequences([sentence])
    # Make length 200
    sequence = pad_sequences(sequence, maxlen=200)
    # Predict
    prediction = model.predict(sequence, verbose=0)[0][0]

    if prediction >= 0.5:
        print("\nPositive Review")
        print("The sentence has a positive sentiment.")
        print("Confidence:", round(prediction * 100, 2), "%")
    else:
        print("\nNegative Review")
        print("The sentence has a negative sentiment.")
        print("Confidence:", round((1 - prediction) * 100, 2), "%")

sentence = input("\nEnter a movie review: ")
predict_sentiment(sentence)

100%|██████████| 25.7M/25.7M [00:00<00:00, 113MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/rehanliaqat17/imbd-dataset/versions/1
Files: ['IMDB_Dataset.csv']
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Dataset size: (50000, 2)
Epoch 1/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 32s 48ms/step - accuracy: 0.6951 - loss: 0.5704
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 31s 49ms/step - accuracy: 0.7904 - loss: 0.4533
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 42s 50ms/step - accuracy: 0.8842 - loss: 0.2882
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.8243 - loss: 0.4298

Test Accuracy: 82.42999911308289 %

Enter a movie review: It was very curious

Positive Review
The sentence has a positive sentiment.
Conf